In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(".env", override=True)
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"



In [3]:
def get_completion(prompt):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0 ))
    return response.text

In [4]:
response = get_completion("What is the capital of France?")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [5]:
print(response)

The capital of France is Paris.


In [6]:
response = get_completion("Take the letters in lollipop \
and reverse them")
print(response)

The word "lollipop" spelled backwards is **popillol**.


In [7]:
response = get_completion("""Take the letters in \
l-o-l-l-i-p-o-p and reverse them""")

In [9]:
response

'Reversing the letters in "l-o-l-l-i-p-o-p" gives you:\n\n**p-o-p-i-l-l-o-l** (popillol)'

In [11]:
def get_completion_from_messages(
    messages,
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_tokens=500):
    
    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )
    
    return response.text

In [21]:
def get_completion_from_messages(
    messages,
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_tokens=500
):
    system_instruction = ""
    contents = []

    for message in messages:
        if message["role"] == "system":
            system_instruction = message["content"]

        elif message["role"] == "user":
            contents.append({
                "role": "user",
                "parts": [{"text": message["content"]}]
            })

        elif message["role"] == "assistant":
            contents.append({
                "role": "model",
                "parts": [{"text": message["content"]}]
            })

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    return response.text

In [22]:
messages = [
    {
        'role': 'system',
        'content': 'All your responses must be one sentence long.'
    },
    {
        'role': 'user',
        'content': 'write me a story about a happy carrot'
    },
]

response = get_completion_from_messages(
    messages,
    temperature=1
)

print(response)

Basking in the warm, rich soil of a sun-drenched garden, Barnaby the carrot smiled widely as he listened to the buzzing bees and happily awaited the farmer's harvest.


In [23]:
# combined
messages =  [  
{'role':'system',
 'content':"""You are an assistant who \
responds in the style of Dr Seuss. \
All your responses must be one sentence long."""},    
{'role':'user',
 'content':"""write me a story about a happy carrot"""},
] 
response = get_completion_from_messages(messages, 
                                        temperature =1)
print(response)

Oh, look at the carrot so bright and so sweet, dancing about on two orange-y feet!


In [28]:
def get_completion_and_token_count(
    messages,
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_tokens=500
):
    system_instruction = ""
    contents = []

    for message in messages:
        if message["role"] == "system":
            system_instruction = message["content"]

        elif message["role"] == "user":
            contents.append({
                "role": "user",
                "parts": [{"text": message["content"]}]
            })

        elif message["role"] == "assistant":
            contents.append({
                "role": "model",
                "parts": [{"text": message["content"]}]
            })

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    content = response.text

    token_dict = {
        "prompt_tokens": response.usage_metadata.prompt_token_count,
        "completion_tokens": response.usage_metadata.candidates_token_count,
        "total_tokens": response.usage_metadata.total_token_count
    }

    return content, token_dict

In [31]:
messages = [
    {
        'role': 'system',
        'content': """You are an assistant who responds
        in the style of Dr Seuss."""
    },
    {
        'role': 'user',
        'content': """write me a very short poem
        about a happy carrot"""
    }
]

response, token_dict = get_completion_and_token_count(messages)

print(response)


Oh, look at the carrot!
So happy and bright!
He jumps up with joy
In the warm morning light!

He wears a green hat
Of a very nice hue,
And he thinks that the world
Is quite wonderful, too!


In [32]:
print(token_dict)

{'prompt_tokens': 30, 'completion_tokens': 53, 'total_tokens': 83}
